In [2]:
from src.models.spatiotemporal_baseline import TGCNModel
from src.dataloading.gnndataloader import GNNDataLoader
from src.utils import get_data_env

modelname   = 'temporal_gcn__'
gg          = 'commuter_t1000_selfmean_rowwise'
name        = modelname + f"{gg}"

disease_name    = 'influenza'
model           = 'tgcn'
nuts_level      = 'nuts3'
min_date        ='2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False


horizon_size    = 1
horizon_leadtime= 3
sequence_length = 12
lags            = 1


# training hparams
n_epochs        = 100
lr              = 0.0005
min_delta       = 0.0001
loss            = 'mse'

global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  :   {'mode': 'min', 'factor': 0.5, 'patience': 7},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }

epidata_loader_basis = GNNDataLoader(disease_name, get_data_env(), nuts_level=nuts_level, min_date=min_date,max_date=max_date, include_population=False, horizon_size = horizon_size, horizon_leadtime = horizon_leadtime, sequence_length=sequence_length, split_berlin=split_berlin)
epidata_loader_basis.add_time_features()
epidata_loader_basis.log_transform_target()
epidata_loader_basis.set_splits(split_trainval, split_valtest)
epidata_loader_basis.normalize()
epidata_loader_basis.add_lagged_features(lags = lags)
epidata_loader_basis.finalize()
dl = epidata_loader_basis.retrieve_graph(gg).construct_dataloaders()

ml = TGCNModel(dl, name = name)
ml.load_config(name)
ml.forecast('test')



Dataloader temporal windowing: extending data collection from 2006-05-15 to 2006-01-30 (+15 weeks)
berlin districts removed


/home/de-schrijvers/projects/germany_gnn/src/models/weights_manager.py:123: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(weights_path, map_location=

✓ Model loaded


Forecasting test: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 37/37 [00:00<00:00, 40.52it/s]

Test loss: 0.9199


In [4]:
print(dl)


        <EpiDataLoader(
        ------------- DATA -----------------
        disease         : influenza,
        features        : ['timestamp_sin', 'timestamp_cos', 'incidence_lag3'],
        nuts_level      : nuts3,
        date_range      : [2006-01-30 - 2020-06-01],
        nodes           : 400,
        data_rows       : 299200

        ------------- TASK -----------------
        horizon size    : 1,
        horizon leadtime: 3,

        ------------- LOAD -----------------
        data stages     : ['context', 'raw', 'processed', 'processed_split', 'normalized', 'final'],
        sequence length : 12
        split summary   : train / val / test: 86.2% / 7.0% / 6.9%
)>
